## Assignment for Module 6

In this assignment you will continue working with the housing price per district from the previous module assignment, this time training SVM models, both for regression and classification.

#### Getting the data for the assignment (similar to the notebook from chapter 2 of Hands-On...)

In [1]:
# Enable NVIDIA cuML 13
%load_ext cuml.accel

In [2]:
import os
import tarfile
from six.moves import urllib

DOWNLOAD_ROOT = "https://raw.githubusercontent.com/ageron/handson-ml/master/"
HOUSING_PATH = os.path.join("datasets", "housing")
HOUSING_URL = DOWNLOAD_ROOT + "datasets/housing/housing.tgz"

def fetch_housing_data(housing_url=HOUSING_URL, housing_path=HOUSING_PATH):
    if not os.path.isdir(housing_path):
        os.makedirs(housing_path)
    tgz_path = os.path.join(housing_path, "housing.tgz")
    urllib.request.urlretrieve(housing_url, tgz_path)
    housing_tgz = tarfile.open(tgz_path)
    housing_tgz.extractall(path=housing_path)
    housing_tgz.close()

In [3]:
fetch_housing_data()

/tmp/ipykernel_32170/2565266754.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tgz.extractall(path=housing_path)


In [4]:
import pandas as pd

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

In [5]:
housing = load_housing_data()

### Fix the categories in the categorical variable

In [6]:
d = {'<1H OCEAN':'LESS_1H_OCEAN', 'INLAND':'INLAND', 'ISLAND':'ISLAND', 'NEAR BAY':'NEAR_BAY', 'NEAR OCEAN':'NEAR_OCEAN'}
housing['ocean_proximity'] = housing['ocean_proximity'].map(lambda s: d[s])

### Add 2 more features

In [7]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["population_per_household"]=housing["population"]/housing["households"]

### Fix missing data

In [8]:
median = housing["total_bedrooms"].median()
# housing["total_bedrooms"].fillna(median, inplace=True)
housing.fillna({"total_bedrooms": median}, inplace=True)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,rooms_per_household,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR_BAY,6.984127,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR_BAY,6.238137,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR_BAY,8.288136,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR_BAY,5.817352,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR_BAY,6.281853,2.181467
...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND,5.045455,2.560606
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND,6.114035,3.122807
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND,5.205543,2.325635
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND,5.329513,2.123209


In [9]:
# Check if imputation worked: result should be 0
print(housing["total_bedrooms"].isna().sum())

0


### Create dummy variables based on the categorical variable

In [10]:
one_hot = pd.get_dummies(housing['ocean_proximity'])
housing = housing.drop('ocean_proximity', axis=1)
housing = housing.join(one_hot)

### Check the data

In [11]:
housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   longitude                 20640 non-null  float64
 1   latitude                  20640 non-null  float64
 2   housing_median_age        20640 non-null  float64
 3   total_rooms               20640 non-null  float64
 4   total_bedrooms            20640 non-null  float64
 5   population                20640 non-null  float64
 6   households                20640 non-null  float64
 7   median_income             20640 non-null  float64
 8   median_house_value        20640 non-null  float64
 9   rooms_per_household       20640 non-null  float64
 10  population_per_household  20640 non-null  float64
 11  INLAND                    20640 non-null  bool   
 12  ISLAND                    20640 non-null  bool   
 13  LESS_1H_OCEAN             20640 non-null  bool   
 14  NEAR_BAY         

### Partition into train and test

Use train_test_split from sklearn.model_selection to partition the dataset into 70% for training and 30% for testing.

You can use the 70% for training set as both training and validation by using cross-validation.


In [12]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(housing, test_size=0.3, random_state=42)

### Features

In [13]:
target = 'median_house_value'
features = list(train_set.columns)
features = [f for f in features if f!=target]

In [14]:
X_tr = train_set[features]
y_tr = train_set[[target]]

X_te = test_set[features]
y_te = test_set[[target]]

### Scaling features

Similarly, use StandardScaler from sklearn.preprocessing to normalize the training and testing data, using the training data

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_tr)
X_tr = scaler.transform(X_tr)
X_te = scaler.transform(X_te)

#### Comparing models

In [16]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import numpy as np

def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())

### Linear regression on original features (no transformations) --- benchmark

In [17]:
from sklearn.linear_model import LinearRegression
lin_scores = cross_val_score(LinearRegression(), train_set[features], train_set[target], scoring="neg_mean_squared_error", cv=4)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

Scores: [70143.94393026 67456.39127249 67318.32588831 70866.26065407]
Mean: 68946.2304362817


### 1. Support Vector Machines for Regression

#### (a) In this exercise your goal is to tune SVR with FBR kernel, and make the average score mean_squared_error over 3-folds (cv=3) below 58000.

You are encouraged to try optimizing any of the hyper-parameters of SVR

See http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html for more details

However, as a hint, you can focus on C and gamma.

Hint 2: if when you try different values for a hyper-parameter, the optimal models corresponds to one of the extreme values in your range, that probably means you can keep improving your solution by considering values beyond the current range.



In [18]:
from sklearn.svm import SVR
from scipy.stats import reciprocal, uniform

# np.logspace(-3, -1, 5) means 5 values in the log scale from 10^-3 (0.001) to 10^-1 (0.1)
param_grid = {"C": np.linspace(1000, 100000, 10), "gamma": np.logspace(-3, -1, 5)}

grid_search_rbf = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_rbf.fit(X_tr, np.ravel(y_tr))

GridSearchCV(cv=3, estimator=SVR(),
             param_grid={'C': array([  1000.,  12000.,  23000.,  34000.,  45000.,  56000.,  67000.,
        78000.,  89000., 100000.]),
                         'gamma': array([0.001     , 0.00316228, 0.01      , 0.03162278, 0.1       ])},
             scoring='neg_mean_squared_error')

In [19]:
# {'C': 1000, 'gamma': 0.01}: 119002
# {'C': 1.0,  'gamma': 0.1}: 118783
# {'C': 1.0,  'gamma': 'scaled'}: 118771
# {'C': 1000000, 'gamma': 0.01}: 115579
# {'C': 1000, 'gamma': 0.03162277660168379}: 75281
# {'C': 100000, 'gamma': 0.1}: 57255.35752847252

print(grid_search_rbf.best_params_)
print(np.sqrt(-grid_search_rbf.best_score_))

{'C': np.float64(100000.0), 'gamma': np.float64(0.1)}
57255.35752847252


**Average MSE < 58,000 found with C = 100,000 and gamma = 0.1**

### Performance on Test Set

In [20]:
# %%cuml.accel.profile
from sklearn.metrics import mean_squared_error

final_model = grid_search_rbf.best_estimator_   ## THIS SHOULD BE THE BEST GRID_SEARCH ##

y_te_estimation = final_model.predict(X_te)

final_mse = mean_squared_error(y_te, y_te_estimation)
final_rmse = np.sqrt(final_mse)
print(final_rmse)

56466.14958603325


**Test set MSE = 56,466**

### 2. SVM for Classification

Now we transform the continuous target into a binary variable, indicating whether or not the price is above the average $179700


In [21]:
from sklearn.metrics import accuracy_score

In [22]:
np.median(housing[['median_house_value']])

np.float64(179700.0)

#### Binary target variable

In [23]:
y_tr_b = 1*np.ravel(y_tr>=179700.0)
y_te_b = 1*np.ravel(y_te>=179700.0)

#### Linear SVM for classification

In [24]:
from sklearn.svm import LinearSVC

In [25]:
lin_clf = LinearSVC(random_state=42)
lin_clf.fit(X_tr, y_tr_b)

LinearSVC(random_state=42)

In [26]:
y_pred = lin_clf.predict(X_tr)
accuracy_score(y_tr_b, y_pred)

0.8370708748615725

**Baseline accuracy = 83.7%**

### (a) Does SVC (with default hyper-parameters) improve the performance of the linear SVM?

In [27]:
from sklearn.svm import SVC

In [28]:
## YOUR CODE HERE ##

# Scale the data (X_tr, y_tr_b, X_te, y_te_b)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr.astype(np.float32))
X_te_scaled = scaler.transform(X_te.astype(np.float32))

In [29]:
# Train SVC
svm_clf = SVC()
svm_clf.fit(X_tr_scaled, y_tr_b)

SVC()

In [30]:
y_pred = svm_clf.predict(X_tr_scaled)
accuracy_score(y_tr_b, y_pred)

0.866140642303433

**SVC accuracy with default hyperparameters on scaled data = 86.6%**

**It was able to achive better performance compared to Linear SVM.**

### (b) Use randomized search to tune hyper-parameters of SVC and improve its performance

In [31]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import reciprocal, uniform

In [35]:
## YOUR CODE HERE ##

# Randomized search with 3-fold CV
param_distributions = {"gamma": reciprocal(0.001, 0.1), "C": uniform(100, 10000)}   # tried (1, 10), (1, 100), (100, 10000)
rnd_search_cv = RandomizedSearchCV(svm_clf, param_distributions, n_iter=10, verbose=2, cv=3)
rnd_search_cv.fit(X_tr_scaled, y_tr_b)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END ...C=2869.8907561521214, gamma=0.003973161343283379; total time=   1.4s
[CV] END ...C=2869.8907561521214, gamma=0.003973161343283379; total time=   1.3s
[CV] END ...C=2869.8907561521214, gamma=0.003973161343283379; total time=   1.5s
[CV] END ...C=7933.679881170663, gamma=0.0037571835706394848; total time=   3.7s
[CV] END ...C=7933.679881170663, gamma=0.0037571835706394848; total time=   3.3s
[CV] END ...C=7933.679881170663, gamma=0.0037571835706394848; total time=   3.5s
[CV] END .....C=4389.76539901043, gamma=0.013730022453467662; total time=   4.0s
[CV] END .....C=4389.76539901043, gamma=0.013730022453467662; total time=   4.1s
[CV] END .....C=4389.76539901043, gamma=0.013730022453467662; total time=   4.0s
[CV] END ....C=1870.7994056380926, gamma=0.01751251357633494; total time=   2.0s
[CV] END ....C=1870.7994056380926, gamma=0.01751251357633494; total time=   2.2s
[CV] END ....C=1870.7994056380926, gamma=0.01751

RandomizedSearchCV(cv=3, estimator=SVC(),
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7995ca897e10>,
                                        'gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7995bc568190>},
                   verbose=2)

In [36]:
rnd_search_cv.best_estimator_

SVC(C=np.float64(1870.7994056380926), gamma=np.float64(0.01751251357633494))

In [37]:
# Accruacy with best estimator (C and Gamma)

y_pred = rnd_search_cv.best_estimator_.predict(X_tr_scaled)
accuracy_score(y_tr_b, y_pred)

0.8939645625692137

**Accuracy after hyperparameter tuning: 89.3%**

Hyperparameter tuning achieved better accuracy on training set with 3-fold CV.

**Predict on test set**

In [39]:
y_pred_test = rnd_search_cv.best_estimator_.predict(X_te_scaled)
accuracy_score(y_te_b, y_pred_test)

0.8761304909560723

**Test set accuracy = 87.6%**

### (c) Train a Logistic Regression (search the best hyper-parameters) and compare its performance with SVC

In [40]:
## YOUR CODE HERE ##

# (X_tr_scaled, y_tr_b, X_te_scaled, y_te_b)

from sklearn.linear_model import LogisticRegression

# Train Logistic Regression model without hyperparameter tuning to get baseline score
lgr_clf = LogisticRegression()
lgr_clf.fit(X_tr_scaled, y_tr_b)

LogisticRegression()

In [41]:
y_pred = lgr_clf.predict(X_tr_scaled)
accuracy_score(y_tr_b, y_pred)

0.8382475083056479

**Logistic regression baseline accuracy = 83.8%**

Let's use randomized search to tune hyperparameters of Logistic Regression and improve its performance.\

In [54]:
# Randomized search with 3-fold CV
param_distributions = {"C": uniform(1, 10000)}
rnd_search_cv_lgr = RandomizedSearchCV(lgr_clf, param_distributions, n_iter=100, verbose=2, cv=3)
rnd_search_cv_lgr.fit(X_tr_scaled, y_tr_b)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END ...............................C=2566.5826270363214; total time=   0.0s
[CV] END ...............................C=2566.5826270363214; total time=   0.0s
[CV] END ...............................C=2566.5826270363214; total time=   0.0s
[CV] END ................................C=2205.190383869411; total time=   0.0s
[CV] END ................................C=2205.190383869411; total time=   0.0s
[CV] END ................................C=2205.190383869411; total time=   0.0s
[CV] END ................................C=6986.134887454768; total time=   0.0s
[CV] END ................................C=6986.134887454768; total time=   0.0s
[CV] END ................................C=6986.134887454768; total time=   0.0s
[CV] END ................................C=2380.237607073018; total time=   0.0s
[CV] END ................................C=2380.237607073018; total time=   0.0s
[CV] END ................................C=238

RandomizedSearchCV(cv=3, estimator=LogisticRegression(), n_iter=100,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7995c8390d50>},
                   verbose=2)

In [55]:
rnd_search_cv_lgr.best_estimator_

LogisticRegression(C=np.float64(432.00745174819434))

In [56]:
# Accruacy with best estimator (C)
y_pred_lgr = rnd_search_cv_lgr.best_estimator_.predict(X_tr_scaled)
accuracy_score(y_tr_b, y_pred_lgr)

0.8383859357696567

**Accuracy did not improve. Let's try tuning additional hyperparameters.**

In [62]:
# Randomized search with 3-fold CV

# solver: liblinear for small datasets, newton-c* for categorical features, lbfgs is default.
param_distributions = {
    "C": uniform(100, 10000),
    "solver": ["lbfgs", "liblinear", "newton-cg", "newton-cholesky"],
    "max_iter": [200]
}
rnd_search_cv_lgr_2 = RandomizedSearchCV(lgr_clf, param_distributions, n_iter=100, verbose=3, cv=3)
rnd_search_cv_lgr_2.fit(X_tr_scaled, y_tr_b)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV 1/3] END C=1310.5628431293746, max_iter=200, solver=newton-cg;, score=0.840 total time=   0.1s
[CV 2/3] END C=1310.5628431293746, max_iter=200, solver=newton-cg;, score=0.837 total time=   0.1s
[CV 3/3] END C=1310.5628431293746, max_iter=200, solver=newton-cg;, score=0.837 total time=   0.1s
[CV 1/3] END C=7188.62237703092, max_iter=200, solver=liblinear;, score=0.840 total time=   0.1s
[CV 2/3] END C=7188.62237703092, max_iter=200, solver=liblinear;, score=0.837 total time=   0.1s
[CV 3/3] END C=7188.62237703092, max_iter=200, solver=liblinear;, score=0.836 total time=   0.0s
[CV 1/3] END C=1347.3629650441076, max_iter=200, solver=newton-cholesky;, score=0.840 total time=   0.1s
[CV 2/3] END C=1347.3629650441076, max_iter=200, solver=newton-cholesky;, score=0.837 total time=   0.1s
[CV 3/3] END C=1347.3629650441076, max_iter=200, solver=newton-cholesky;, score=0.836 total time=   0.2s
[CV 1/3] END C=5627.033805787248, 

RandomizedSearchCV(cv=3, estimator=LogisticRegression(), n_iter=100,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x799569d99710>,
                                        'max_iter': [200],
                                        'solver': ['lbfgs', 'liblinear',
                                                   'newton-cg',
                                                   'newton-cholesky']},
                   verbose=3)

In [63]:
rnd_search_cv_lgr_2.best_estimator_

LogisticRegression(C=np.float64(1310.5628431293746), max_iter=200,
                   solver='newton-cg')

In [64]:
# Accruacy with best estimator (C and solver)
y_pred_lgr = rnd_search_cv_lgr_2.best_estimator_.predict(X_tr_scaled)
accuracy_score(y_tr_b, y_pred_lgr)

0.8383859357696567

**Hyperparameter tuning did not produce a better training accuracy.**

Predict on test set

In [65]:
y_pred_test = rnd_search_cv_lgr_2.best_estimator_.predict(X_te_scaled)
accuracy_score(y_te_b, y_pred_test)

0.8407622739018088

Test set accuracy = 84%

**Verdict: Logistic regression performed worse compared to SVC.**